# 07 - Stock Recommendation Engine

## Objective

This notebook converts a 14-day demand forecast into simple stock recommendations. It connects latest product information, future features, forecast aggregation, current stock, supplier lead time and safety stock.

**Inputs:** processed daily data, modeling data, XGBoost pipeline and evaluation summary  
**Outputs:** `reports/stock_recommendations.csv` and `reports/stock_recommendation_summary.csv`

## Imports and Load Inputs

In [1]:
import os
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

import sys
import joblib
import numpy as np

sys.path.append(str(ROOT))
from src.recommendation import generate_stock_recommendations

MODELS_PATH = ROOT / "artifacts" / "models"
model = joblib.load(MODELS_PATH / "xgboost_model.pkl")
df_model = pd.read_csv(ROOT / "data" / "processed" / "modeling_dataset.csv", parse_dates=["sale_date"])
df_daily = pd.read_csv(ROOT / "data" / "processed" / "daily_product_sales.csv", parse_dates=["sale_date"])
evaluation_summary = pd.read_csv(ROOT / "reports" / "model_evaluation_summary.csv")

forecast_horizon_days = 14
assert evaluation_summary.sort_values("MAE").iloc[0]["model_name"] == "XGBoost"
print("Forecast horizon:", forecast_horizon_days, "days")

Forecast horizon: 14 days


## Get Latest Product State and Future Dates

The latest valid product price is carried into the forecast as the previous known price. Weather is not used by the model because this project does not have future weather forecasts.

In [2]:
last_historical_date = df_daily["sale_date"].max()
future_dates = pd.date_range(last_historical_date + pd.Timedelta(days=1), periods=forecast_horizon_days, freq="D")

latest_price = (
    df_daily.dropna(subset=["unit_price_brl"])
    .sort_values(["product_name", "sale_date"])
    .groupby("product_name")
    .tail(1)
    .set_index("product_name")["unit_price_brl"]
)
print("Last historical date:", last_historical_date.date())
print("Future period:", future_dates.min().date(), "to", future_dates.max().date())

Last historical date: 2026-05-29
Future period: 2026-05-30 to 2026-06-12


## Build Future Features and Generate Demand Forecast

Forecasts are generated one future day at a time for each product. The first day uses actual history. Later days use earlier non-negative predictions as history. This keeps 1-, 7- and 14-day lags aligned with each future date and avoids using unknown future demand.

Daily predictions remain decimal values. They are not rounded before the 14-day aggregation.

In [3]:
model_feature_columns = [
    "unit_price_1d", "day_of_week", "day_of_month", "month",
    "week_of_year", "is_weekend", "quantity_1d", "quantity_7d",
    "quantity_14d", "rolling_1d", "rolling_7d", "rolling_14d",
    "product_name",
]

assert model_feature_columns == list(model.feature_names_in_)
assert set(model_feature_columns).issubset(df_model.columns)

future_rows = []
for product_name, product_history in df_daily.groupby("product_name"):
    product_history = product_history.sort_values("sale_date")
    demand_history = product_history["quantity_sold"].astype(float).tolist()

    for future_date in future_dates:
        row = {
            "unit_price_1d": float(latest_price.loc[product_name]),
            "day_of_week": future_date.dayofweek,
            "day_of_month": future_date.day,
            "month": future_date.month,
            "week_of_year": int(future_date.isocalendar().week),
            "is_weekend": int(future_date.dayofweek in [5, 6]),
            "quantity_1d": demand_history[-1],
            "quantity_7d": demand_history[-7],
            "quantity_14d": demand_history[-14],
            "rolling_1d": float(np.mean(demand_history[-1:])),
            "rolling_7d": float(np.mean(demand_history[-7:])),
            "rolling_14d": float(np.mean(demand_history[-14:])),
            "product_name": product_name,
        }
        forecast_raw = float(model.predict(pd.DataFrame([row])[model_feature_columns])[0])
        forecast_non_negative = max(forecast_raw, 0.0)

        future_rows.append({
            "product_name": product_name,
            "sale_date": future_date,
            "forecast_raw": forecast_raw,
            "forecast_non_negative": forecast_non_negative,
            **row,
        })
        demand_history.append(forecast_non_negative)

future_forecast_df = pd.DataFrame(future_rows)

expected_rows = df_daily["product_name"].nunique() * forecast_horizon_days
assert len(future_forecast_df) == expected_rows
assert not future_forecast_df.duplicated(["product_name", "sale_date"]).any()
assert future_forecast_df[model_feature_columns].notna().all().all()
assert (future_forecast_df["forecast_non_negative"] >= 0).all()

future_forecast_df[["product_name", "sale_date", "forecast_raw", "forecast_non_negative"]].head(14)

,product_name,sale_date,forecast_raw,forecast_non_negative
0,Bag Delivery 45L,2026-05-30,2.529277,2.529277
1,Bag Delivery 45L,2026-05-31,2.493765,2.493765
2,Bag Delivery 45L,2026-06-01,4.477439,4.477439
3,Bag Delivery 45L,2026-06-02,3.468667,3.468667
4,Bag Delivery 45L,2026-06-03,3.260210,3.260210
5,Bag Delivery 45L,2026-06-04,4.346901,4.346901
6,Bag Delivery 45L,2026-06-05,2.149912,2.149912
7,Bag Delivery 45L,2026-06-06,3.955084,3.955084
8,Bag Delivery 45L,2026-06-07,1.823315,1.823315
9,Bag Delivery 45L,2026-06-08,4.472377,4.472377


## Aggregate the 14-Day Forecast by Product

In [4]:
forecast_by_product_df = (
    future_forecast_df.groupby("product_name", as_index=False)
    .agg(
        forecast_horizon_days=("sale_date", "nunique"),
        forecasted_demand_raw=("forecast_raw", "sum"),
        forecasted_demand_non_negative=("forecast_non_negative", "sum"),
    )
)
forecast_by_product_df["forecasted_demand_units"] = np.ceil(
    forecast_by_product_df["forecasted_demand_non_negative"]
).astype(int)

assert (forecast_by_product_df["forecast_horizon_days"] == forecast_horizon_days).all()
forecast_by_product_df.head()

,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units
0,Bag Delivery 45L,14,47.962948,47.962948,48
1,Bag Delivery 80L,14,24.289850,24.289850,25
2,Balaclava,14,14.001884,14.001884,15
3,Baú Moto 30L,14,19.775357,19.775357,20
4,Capa de Chuva,14,37.589008,37.589008,38


## Get Current Stock and Supplier Lead Time

Current stock and lead time use the latest valid product snapshot. Missing stock is filled with zero as a conservative assumption. Missing lead time uses a 7-day default.

In [5]:
latest_business_state = (
    df_daily.sort_values(["product_name", "sale_date"])
    .groupby("product_name", as_index=False)
    .tail(1)[["product_name", "current_stock_snapshot", "supplier_lead_time_days"]]
)

current_stock_df = latest_business_state[["product_name", "current_stock_snapshot"]].rename(
    columns={"current_stock_snapshot": "current_stock"}
)
lead_time_df = latest_business_state[["product_name", "supplier_lead_time_days"]].copy()

products_without_stock = current_stock_df["current_stock"].isna().sum()
products_without_lead_time = lead_time_df["supplier_lead_time_days"].isna().sum()
print("Products with stock filled as zero:", int(products_without_stock))
print("Products with default lead time:", int(products_without_lead_time))

Products with stock filled as zero: 0
Products with default lead time: 0


## Generate Final Recommendations

`generate_stock_recommendations` is kept in `src/recommendation.py` because it is reusable business logic. It returns one row per product after these steps:

1. Safety stock = 20% of rounded-up 14-day demand.
2. Required stock = forecasted demand + safety stock.
3. Recommended purchase = required stock - current stock, with a minimum of zero.
4. Status = critical, warning, healthy or overstock using the documented stock thresholds.
5. Priority score = purchase quantity × lead time × status weight, scaled from 0 to 100.

In [6]:
stock_recommendations_df = generate_stock_recommendations(
    forecast_by_product_df=forecast_by_product_df,
    current_stock_df=current_stock_df,
    lead_time_df=lead_time_df,
    default_lead_time_days=7,
    safety_stock_pct=0.20,
    overstock_multiplier=1.50,
)

stock_recommendations_df.head(10)

,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units,current_stock,safety_stock,required_stock,recommended_purchase_quantity,supplier_lead_time_days,stock_status,priority_score
0,Suporte Celular Moto,14,50.562013,50.562013,51,60,11,62,2,3,warning,100
1,Bag Delivery 45L,14,47.962948,47.962948,48,109,10,58,0,5,overstock,0
2,Bag Delivery 80L,14,24.289850,24.289850,25,108,5,30,0,9,overstock,0
3,Balaclava,14,14.001884,14.001884,15,59,3,18,0,10,overstock,0
4,Baú Moto 30L,14,19.775357,19.775357,20,63,4,24,0,5,overstock,0
5,Capa de Chuva,14,37.589008,37.589008,38,104,8,46,0,6,overstock,0
6,Capacete LS2,14,6.045690,6.045690,7,38,2,9,0,10,overstock,0
7,Capacete Pro Tork,14,28.329035,28.329035,29,100,6,35,0,4,overstock,0
8,Carregador USB Moto,14,12.844502,12.844502,13,95,3,16,0,8,overstock,0
9,Intercomunicador,14,3.297890,3.297890,4,94,1,5,0,6,overstock,0


## Validate and Save Reports

In [7]:
integer_columns = [
    "forecast_horizon_days", "forecasted_demand_units", "current_stock",
    "safety_stock", "required_stock", "recommended_purchase_quantity",
    "supplier_lead_time_days", "priority_score",
]
key_columns = ["product_name", "stock_status"] + integer_columns

assert stock_recommendations_df["product_name"].is_unique
assert len(stock_recommendations_df) == df_daily["product_name"].nunique()
assert stock_recommendations_df[key_columns].notna().all().all()
assert (stock_recommendations_df["forecast_horizon_days"] == forecast_horizon_days).all()
for column in integer_columns:
    assert pd.api.types.is_integer_dtype(stock_recommendations_df[column])
for column in ["current_stock", "safety_stock", "required_stock", "recommended_purchase_quantity"]:
    assert (stock_recommendations_df[column] >= 0).all()

REPORTS_PATH = ROOT / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)
stock_recommendations_df.to_csv(REPORTS_PATH / "stock_recommendations.csv", index=False, encoding="utf-8")

status_counts = stock_recommendations_df["stock_status"].astype(str).value_counts()
highest_priority_product = stock_recommendations_df.sort_values(
    ["priority_score", "recommended_purchase_quantity"], ascending=False
).iloc[0]["product_name"]

recommendation_summary_df = pd.DataFrame([{
    "total_products": stock_recommendations_df["product_name"].nunique(),
    "critical_products": int(status_counts.get("critical", 0)),
    "warning_products": int(status_counts.get("warning", 0)),
    "healthy_products": int(status_counts.get("healthy", 0)),
    "overstock_products": int(status_counts.get("overstock", 0)),
    "total_recommended_purchase_units": int(stock_recommendations_df["recommended_purchase_quantity"].sum()),
    "highest_priority_product": highest_priority_product,
    "forecast_horizon_days": forecast_horizon_days,
    "selected_model": "XGBoost",
}])
recommendation_summary_df.to_csv(REPORTS_PATH / "stock_recommendation_summary.csv", index=False, encoding="utf-8")

assert int(recommendation_summary_df.loc[0, "total_products"]) == len(stock_recommendations_df)
assert int(recommendation_summary_df.loc[0, "total_recommended_purchase_units"]) == int(stock_recommendations_df["recommended_purchase_quantity"].sum())
print("Saved recommendation reports to:", REPORTS_PATH.resolve())
recommendation_summary_df

Saved recommendation reports to: C:\DEV\motostock-ai\reports


,total_products,critical_products,warning_products,healthy_products,overstock_products,total_recommended_purchase_units,highest_priority_product,forecast_horizon_days,selected_model
0,12,0,1,0,11,2,Suporte Celular Moto,14,XGBoost


## Business Findings

In [8]:
products_to_restock = stock_recommendations_df[stock_recommendations_df["recommended_purchase_quantity"] > 0]
print("Products requiring replenishment:", len(products_to_restock))
print("Critical products:", int((stock_recommendations_df["stock_status"].astype(str) == "critical").sum()))
print("Highest-priority product:", highest_priority_product)
print("Total recommended purchase units:", int(stock_recommendations_df["recommended_purchase_quantity"].sum()))
display(stock_recommendations_df.head(10))

Products requiring replenishment: 1
Critical products: 0
Highest-priority product: Suporte Celular Moto
Total recommended purchase units: 2


,product_name,forecast_horizon_days,forecasted_demand_raw,forecasted_demand_non_negative,forecasted_demand_units,current_stock,safety_stock,required_stock,recommended_purchase_quantity,supplier_lead_time_days,stock_status,priority_score
0,Suporte Celular Moto,14,50.562013,50.562013,51,60,11,62,2,3,warning,100
1,Bag Delivery 45L,14,47.962948,47.962948,48,109,10,58,0,5,overstock,0
2,Bag Delivery 80L,14,24.289850,24.289850,25,108,5,30,0,9,overstock,0
3,Balaclava,14,14.001884,14.001884,15,59,3,18,0,10,overstock,0
4,Baú Moto 30L,14,19.775357,19.775357,20,63,4,24,0,5,overstock,0
5,Capa de Chuva,14,37.589008,37.589008,38,104,8,46,0,6,overstock,0
6,Capacete LS2,14,6.045690,6.045690,7,38,2,9,0,10,overstock,0
7,Capacete Pro Tork,14,28.329035,28.329035,29,100,6,35,0,4,overstock,0
8,Carregador USB Moto,14,12.844502,12.844502,13,95,3,16,0,8,overstock,0
9,Intercomunicador,14,3.297890,3.297890,4,94,1,5,0,6,overstock,0


## Conclusion and Current Limitations

The engine produces one recommendation per product for the next 14 days. Daily predictions are kept as decimals until product-level aggregation, and negative predictions are set to zero. The final report is sorted by stock risk, priority score and purchase quantity.

This remains a learning project, not a production purchasing system:

- Weather is not used as a model feature because no future weather forecast is available.
- Future price is held at the latest known product price.
- Recursive forecasts can accumulate error across the 14-day horizon.
- Safety stock uses a fixed 20% rule instead of demand variability or service-level calculations.
- Stock and supplier lead time are latest available snapshots, not real-time operational data.
- Missing stock would be filled with zero, and missing lead time would use 7 days.
- Recommendations should be reviewed before purchase decisions.